In [ ]:


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [3]:
!pip install datasets transformers accelerate evaluate librosa jiwer peft bitsandbytes --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.5 MB/s eta 0:00:00:00:01


In [4]:
from datasets import load_dataset

In [ ]:
import torch
import numpy as np
import os
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_from_disk
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    BitsAndBytesConfig
)
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
import evaluate

✅ GPUs visible: 0


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
BASE_DIR       = "/workspace/Fine-Tune-Whisper-small-On-Darija-using-A100-GPU"
DODA_PATH      = f"{BASE_DIR}/data/DODa/data/*.parquet"
PROCESSED_PATH = f"{BASE_DIR}/data/processed/whisper-darija-merged"
OUTPUT_DIR     = f"{BASE_DIR}/models/whisper-darija-qlora-final"

# ✅ Create folders if they don't exist
os.makedirs(f"{BASE_DIR}/data/processed", exist_ok=True)
os.makedirs(f"{BASE_DIR}/models",         exist_ok=True)

In [ ]:
import multiprocessing
print(f"✅ CPUs available: {multiprocessing.cpu_count()}")

In [ ]:
import torch

# ⚡ SPEED BOOST for A100
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-medium",
    language="arabic",
    task="transcribe"
)
print("✅ Processor loaded")

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

✅ Processor loaded


## Find the words written lore than once differently

### First Data preprocessing and Filtering not enough still needs improvements

In [ ]:
import glob
import re
import numpy as np
import librosa
from collections import Counter, defaultdict
from datasets import load_dataset
import pandas as pd

In [ ]:
import glob
import re
import numpy as np
import librosa
from datasets import load_dataset, concatenate_datasets

In [ ]:


print("📥 Loading Dataset 1 (DODa)...")

parquet_files = glob.glob(DODA_PATH)

print(f"📁 Found {len(parquet_files)} parquet files:")
for f in parquet_files:
    print(f"   {f}")

dataset1 = load_dataset("parquet", data_files={"train": parquet_files})

print(f"✅ Dataset 1: {len(dataset1['train']):,} samples")
print(f"   Columns: {dataset1['train'].column_names}")

📥 Loading Dataset 1 (DODa)...
📁 Found 5 parquet files:
   /kaggle/input/datasets/youneseloiarm/moroccan-darija-voice-dataset/DODa/data/train-00002-of-00005.parquet
   /kaggle/input/datasets/youneseloiarm/moroccan-darija-voice-dataset/DODa/data/train-00000-of-00005.parquet
   /kaggle/input/datasets/youneseloiarm/moroccan-darija-voice-dataset/DODa/data/train-00003-of-00005.parquet
   /kaggle/input/datasets/youneseloiarm/moroccan-darija-voice-dataset/DODa/data/train-00001-of-00005.parquet
   /kaggle/input/datasets/youneseloiarm/moroccan-darija-voice-dataset/DODa/data/train-00004-of-00005.parquet


Generating train split: 0 examples [00:00, ? examples/s]

✅ Dataset 1: 12,743 samples
   Columns: ['audio', 'darija_Latn', 'darija_Arab_new', 'english', 'darija_Arab_old']


In [ ]:
def normalize_ar(text):
    if not text:
        return ""
    text = text.replace("أ", "ا")
    text = text.replace("إ", "ا")
    text = text.replace("آ", "ا")
    text = text.replace("ّ", "")   # shadda
    text = text.replace("َ", "")   # fatha
    text = text.replace("ُ", "")   # damma
    text = text.replace("ِ", "")   # kasra
    text = text.replace("ْ", "")   # sukun
    text = text.replace("ً", "")   # tanwin fath
    text = text.replace("ٌ", "")   # tanwin damm
    text = text.replace("ٍ", "")   # tanwin kasr
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
def preprocess(example):
    audio = example["audio"]

    # ── Handle both formats ──
    if hasattr(audio, "get_all_samples"):
        samples = audio.get_all_samples()
        audio_array = samples.data.numpy().squeeze()
        sampling_rate = samples.sample_rate
    else:
        audio_array = np.array(audio["array"], dtype=np.float32)
        sampling_rate = audio["sampling_rate"]

    # ── Resample to 16kHz ──
    if sampling_rate != 16000:
        audio_array = librosa.resample(
            audio_array.astype(np.float32),
            orig_sr=sampling_rate,
            target_sr=16000
        )
    audio_array = audio_array.astype(np.float32)
    audio_array = audio_array / np.max(np.abs(audio_array) + 1e-6)

    # ── Audio length filter ──
    duration_sec = len(audio_array) / 16000
    if duration_sec < 3 or duration_sec > 31.0:
        return {"input_features": None, "labels": None}

    # ── Audio → mel spectrogram ──
    input_features = processor.feature_extractor(
        audio_array,
        sampling_rate=16000
    ).input_features[0]

    # ── Text normalization + tokenization ──
    # Both datasets now use "text" column (unified in load step)
    text = normalize_ar(example["text"])

    if not text:
        return {"input_features": None, "labels": None}

    labels = processor.tokenizer(
        text,
        language="arabic",
        task="transcribe"
    ).input_ids

    # ── Label length filter ──
    if len(labels) < 2 or len(labels) > 448:
        return {"input_features": None, "labels": None}

    return {
        "input_features": input_features,
        "labels":         labels
    }

In [ ]:
print("📥 Loading DODa dataset...")
parquet_files = glob.glob(DODA_PATH)
print(f"📁 Found {len(parquet_files)} parquet files")

doda_raw = load_dataset("parquet", data_files={"train": parquet_files})

# ── Rename to unified column name "text" ──
doda_raw = doda_raw["train"].rename_column("darija_Arab_new", "text")

# ── Remove DODa-specific columns not in dataset2 ──
cols_to_remove = [c for c in doda_raw.column_names
                  if c not in ["audio", "text"]]
doda_raw = doda_raw.remove_columns(cols_to_remove)

# ── Filter empty text ──
doda_raw = doda_raw.filter(
    lambda x: x["text"] is not None
    and x["text"].strip() != ""
    and len(normalize_ar(x["text"])) > 2,
    desc="Filtering DODa bad text"
)
print(f"✅ DODa after filter: {len(doda_raw):,} samples")

# ── Preprocess ──
print("🔄 Preprocessing DODa (10-15 mins)...")
doda_processed = doda_raw.map(
    preprocess,
    remove_columns=doda_raw.column_names,
    num_proc=8,
    writer_batch_size=200,
    desc="Preprocessing DODa"
)

# ── Remove invalid ──
before = len(doda_processed)
doda_processed = doda_processed.filter(
    lambda x: x["input_features"] is not None and x["labels"] is not None,
    desc="Cleaning DODa"
)
print(f"✅ DODa final: {len(doda_processed):,} samples  (removed {before - len(doda_processed):,})")

In [ ]:
print("📥 Loading Dataset 2 (adiren7)...")

ds2_default = load_dataset("adiren7/darija_speech_to_text", "default")
ds2_dr      = load_dataset("adiren7/darija_speech_to_text", "dr")

print(f"   default: {len(ds2_default['train']):,} | dr: {len(ds2_dr['train']):,}")
print(f"   default columns: {ds2_default['train'].column_names}")
print(f"   dr columns:      {ds2_dr['train'].column_names}")

# ── Merge both subsets ──
ds2_raw = concatenate_datasets([
    ds2_default["train"],
    ds2_dr["train"]
])
print(f"✅ Dataset 2 merged: {len(ds2_raw):,} samples")

# ── Keep only audio + text columns ──
# text column is already called "text" in dataset2
cols_to_remove = [c for c in ds2_raw.column_names
                  if c not in ["audio", "text"]]
ds2_raw = ds2_raw.remove_columns(cols_to_remove)

# ── Filter empty text ──
ds2_raw = ds2_raw.filter(
    lambda x: x["text"] is not None
    and x["text"].strip() != ""
    and len(normalize_ar(x["text"])) > 2,
    desc="Filtering Dataset2 bad text"
)
print(f"✅ Dataset 2 after filter: {len(ds2_raw):,} samples")

# ── Preprocess ──
print("🔄 Preprocessing Dataset 2...")
ds2_processed = ds2_raw.map(
    preprocess,
    remove_columns=ds2_raw.column_names,
    num_proc=8,
    writer_batch_size=200,
    desc="Preprocessing Dataset2"
)

# ── Remove invalid ──
before = len(ds2_processed)
ds2_processed = ds2_processed.filter(
    lambda x: x["input_features"] is not None and x["labels"] is not None,
    desc="Cleaning Dataset2"
)
print(f"✅ Dataset 2 final: {len(ds2_processed):,} samples  (removed {before - len(ds2_processed):,})")

In [ ]:
# ── Merge both datasets ──
print("🔗 Merging DODa + Dataset 2...")
full_dataset = concatenate_datasets([doda_processed, ds2_processed])
full_dataset = full_dataset.shuffle(seed=42)
print(f"✅ Total merged samples: {len(full_dataset):,}")

# ── Save to disk ──
print("💾 Saving merged dataset...")
full_dataset.save_to_disk(PROCESSED_PATH)
print(f"✅ Saved to {PROCESSED_PATH}")

# ── Train / Test split ──
dataset = full_dataset.train_test_split(test_size=0.1, seed=42)
print(f"\n✅ Train: {len(dataset['train']):,}")
print(f"✅ Test:  {len(dataset['test']):,}")
print(f"✅ Total: {len(full_dataset):,}")

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

# ✅ Load full model (NO quantization)
model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-medium",
    torch_dtype=torch.float16,   # mixed precision (fast + stable)
    device_map={"": "cuda:0"}
)

# ✅ Important configs for training
model.config.use_cache = False
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.language = "arabic"    # ✅ missing — forces Arabic output
model.generation_config.task = "transcribe" 

print(f"✅ Model loaded on: {next(model.parameters()).device}")

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

✅ Model loaded on: cuda:0


In [19]:
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=[
        "q_proj",      # all q projections (encoder + decoder self_attn + encoder_attn)
        "k_proj",      # all k projections
        "v_proj",      # all v projections
        "out_proj",    # all output projections
        "fc1",         # feed-forward layer 1
        "fc2",         # feed-forward layer 2
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# ```

# This will target **all 12 encoder layers + all 12 decoder layers** including the critical `encoder_attn` (cross-attention) layers automatically — because PEFT sees `encoder_attn.q_proj` and matches it to `q_proj`.

# **Expected output:**
# ```
# trainable params: ~15,000,000 || all params: ~244,000,000 || trainable%: ~6-7%

trainable params: 25,952,256 || all params: 267,687,168 || trainable%: 9.6950


In [20]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("✅ Data collator ready")

✅ Data collator ready


In [21]:
import evaluate
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # ✅ Normalize before computing WER
    pred_str  = [normalize_ar(t) for t in pred_str]
    label_str = [normalize_ar(t) for t in label_str]

    wer = metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 4)}

print("✅ Metric ready")

✅ Metric ready


In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=f"{BASE_DIR}/models/whisper-darija-a100",  # ✅ fixed

    per_device_train_batch_size=16,      # ✅ safer
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,       # ✅ effective batch = 32

    num_train_epochs=3,
    learning_rate=3e-5,

    fp16=True,
    gradient_checkpointing=True,

    dataloader_num_workers=4,            # ✅ added
    dataloader_pin_memory=True,          # ✅ added

    evaluation_strategy="steps",         # ✅ fixed
    eval_steps=500,
    save_steps=500,
    save_total_limit=3,                  # ✅ added

    logging_steps=25,

    predict_with_generate=True,
    generation_max_length=225,
    generation_num_beams=2,              # ✅ added

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    remove_unused_columns=False,
    label_names=["labels"],
)
print("✅ Training args ready")


✅ Training args ready


In [23]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ Trainer ready 🚀 Starting training...")
trainer.train()

✅ Trainer ready 🚀 Starting training...


Step,Training Loss,Validation Loss,Wer
500,0.775217,0.409636,1.556800
1000,0.450879,0.332796,1.776700
1500,0.230948,0.326179,1.743900
2000,0.143197,0.336320,1.801400


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, 

KeyboardInterrupt: 

In [ ]:


model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"✅ Saved to {OUTPUT_DIR}")